# Logistic Regression Baseline

Notebook này chạy trên Google Colab để huấn luyện và đánh giá **Logistic Regression** cho bài toán phân loại ảnh thật và ảnh AI-generated.

Notebook chỉ dùng một model:

```python
sklearn.linear_model.LogisticRegression
```

Không thêm ResNet, Swin, CNNSpot, CLIP hay model khác.

## 1. Cấu hình

Chỉnh các biến ở cell này trước khi chạy. Không dùng đường dẫn Windows trong Colab.

`EXPERIMENT_CASE`:

- `cross_generator`: TH1, train riêng từng generator.
- `combined`: TH2, train trên toàn bộ split `train`, test trên split `val`/`test`.

In [2]:
from pathlib import Path
import json
import logging
import random
import time
from typing import Any, Dict, List, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler


PROJECT_ROOT = Path("/content/drive/MyDrive/genimage_logistic_regression")
DATA_ROOT = None  # Sẽ được set tự động sau khi tải dataset bằng kagglehub.
METADATA_ROOT = Path("/content/drive/MyDrive/genimage_prepared_data/metadata")
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
REPORT_ROOT = PROJECT_ROOT / "report"
DATASET_SLUG = "yangsangtai/tiny-genimage"
EXPERIMENT_CASE = "cross_generator"
RUN_EXPERIMENT_CASES = ["cross_generator", "combined"]

# Chỉ dùng khi EXPERIMENT_CASE = "cross_generator".
# Nếu đặt None, notebook sẽ train riêng từng generator tìm thấy trong metadata.
BASE_GENERATOR = "imagenet_ai_0419_biggan"

FEATURE_EXTRACTOR = "torch_downsample_rgb"
IMAGE_SIZE = 64
BATCH_SIZE = 128
NUM_WORKERS = 2
RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Logistic Regression parameters.
LR_C = 1.0
LR_MAX_ITER = 1000
LR_SOLVER = "lbfgs"
LR_CLASS_WEIGHT = "balanced"
LR_RANDOM_STATE = RANDOM_SEED

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("logistic_regression_baseline")

print("DEVICE:", DEVICE)
print("EXPERIMENT_CASE:", EXPERIMENT_CASE)
print("FEATURE_EXTRACTOR:", FEATURE_EXTRACTOR)

DEVICE: cuda
EXPERIMENT_CASE: cross_generator
FEATURE_EXTRACTOR: torch_downsample_rgb


## 2. Mount Google Drive

In [3]:
from google.colab import drive
from pathlib import Path
import shutil
import os

mount_point = Path("/content/drive")

# Nếu mount cũ bị kẹt, thử unmount trước.
if mount_point.exists():
    try:
        drive.flush_and_unmount()
        print("Đã unmount Drive cũ.")
    except Exception as exc:
        print("Không unmount được hoặc Drive chưa mount:", exc)

# Xóa mount point nếu là folder rỗng/cũ.
try:
    if mount_point.exists() and not any(mount_point.iterdir()):
        shutil.rmtree(mount_point)
except Exception as exc:
    print("Bỏ qua bước xóa mount point:", exc)

drive.mount("/content/drive", force_remount=True)

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("METADATA_ROOT:", METADATA_ROOT)

Đã unmount Drive cũ.
Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/genimage_logistic_regression
DATA_ROOT: None
METADATA_ROOT: /content/drive/MyDrive/genimage_prepared_data/metadata


## 3. Tải dataset ảnh từ Kaggle

Metadata được lưu trên Google Drive, nhưng ảnh gốc Tiny-GenImage không được lưu trong Drive ở bước chuẩn bị dữ liệu.

Vì vậy notebook này tải lại dataset bằng `kagglehub`, sau đó tự đặt `DATA_ROOT` là thư mục cha chứa các folder generator như:

```text
imagenet_ai_0419_biggan/
imagenet_ai_0419_vqdm/
imagenet_glide/
imagenet_midjourney/
```

In [4]:
try:
    import kagglehub
except ImportError:
    !pip install -q kagglehub
    import kagglehub


def find_dataset_root(download_root: Path) -> Path:
    expected_generator = "imagenet_ai_0419_biggan"

    if (download_root / expected_generator).exists():
        return download_root

    candidates = [p for p in download_root.rglob(expected_generator) if p.is_dir()]

    if len(candidates) == 1:
        return candidates[0].parent

    if len(candidates) > 1:
        print("Tìm thấy nhiều thư mục generator candidate:")
        for candidate in candidates:
            print(" -", candidate)
        return candidates[0].parent

    raise FileNotFoundError(
        f"Không tìm thấy {expected_generator} bên trong {download_root}. "
        "Dataset tải về có thể sai cấu trúc hoặc chưa tải thành công."
    )


print(f"Đang tải dataset {DATASET_SLUG} bằng kagglehub...")
download_path = kagglehub.dataset_download(DATASET_SLUG)
DATA_ROOT = find_dataset_root(Path(download_path))

print("DATA_ROOT đã set:", DATA_ROOT)
print("Các thư mục generator tìm thấy:")
for item in sorted(DATA_ROOT.iterdir()):
    if item.is_dir():
        print(" -", item.name)

Đang tải dataset yangsangtai/tiny-genimage bằng kagglehub...
Using Colab cache for faster access to the 'tiny-genimage' dataset.
DATA_ROOT đã set: /kaggle/input/tiny-genimage
Các thư mục generator tìm thấy:
 - imagenet_ai_0419_biggan
 - imagenet_ai_0419_vqdm
 - imagenet_ai_0424_sdv5
 - imagenet_ai_0424_wukong
 - imagenet_ai_0508_adm
 - imagenet_glide
 - imagenet_midjourney


## 4. Hàm tiện ích

In [5]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def timestamp() -> str:
    return time.strftime("%Y%m%d_%H%M%S")


def require_file(path: Path, message: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{message}: {path}")


def save_json(data: Dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def make_output_dir(experiment_case: str, base_generator: str | None) -> Path:
    base_name = base_generator if base_generator else "all_generators"
    run_dir = OUTPUT_ROOT / experiment_case / base_name / timestamp()
    for sub in ["checkpoints", "features", "predictions", "metrics", "plots"]:
        (run_dir / sub).mkdir(parents=True, exist_ok=True)
    return run_dir


set_seed(RANDOM_SEED)

## 5. Load metadata

In [6]:
REQUIRED_COLUMNS = ["image_path", "generator", "original_split", "label", "label_name"]


def resolve_image_path(row: pd.Series) -> str:
    raw_path = Path(str(row["image_path"]))
    if raw_path.exists():
        return str(raw_path)

    if "relative_path" in row and pd.notna(row["relative_path"]):
        candidate = DATA_ROOT / str(row["relative_path"])
        if candidate.exists():
            return str(candidate)

    return str(raw_path)


def read_metadata_csv(path: Path) -> pd.DataFrame:
    require_file(path, "Không tìm thấy file metadata")
    df = pd.read_csv(path)
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Metadata thiếu cột bắt buộc {missing}: {path}")

    df = df.copy()
    df["image_path"] = df.apply(resolve_image_path, axis=1)
    df["label"] = df["label"].astype(int)
    df["generator"] = df["generator"].astype(str)
    df["original_split"] = df["original_split"].astype(str)
    return df


def validate_metadata(train_df: pd.DataFrame, test_df: pd.DataFrame) -> Dict[str, Any]:
    issues: Dict[str, Any] = {}

    for name, df in [("train", train_df), ("test", test_df)]:
        if df.empty:
            raise ValueError(f"Metadata {name} bị rỗng.")

        invalid_labels = sorted(set(df["label"].unique()) - {0, 1})
        if invalid_labels:
            raise ValueError(f"Metadata {name} có label không hợp lệ: {invalid_labels}")

        missing_images = df[~df["image_path"].map(lambda x: Path(str(x)).exists())]
        issues[f"{name}_missing_images"] = int(len(missing_images))
        if len(missing_images) > 0:
            display(missing_images.head())
            raise FileNotFoundError(
                f"Có {len(missing_images)} ảnh không tồn tại trong metadata {name}. "
                "Hãy kiểm tra DATA_ROOT hoặc image_path/relative_path."
            )

        issues[f"{name}_duplicate_rows"] = int(df.duplicated().sum())

    overlap_paths = set(train_df["image_path"].astype(str)) & set(test_df["image_path"].astype(str))
    issues["image_path_overlap"] = int(len(overlap_paths))

    if "sha256" in train_df.columns and "sha256" in test_df.columns:
        overlap_sha = set(train_df["sha256"].dropna().astype(str)) & set(test_df["sha256"].dropna().astype(str))
        issues["sha256_overlap"] = int(len(overlap_sha))
    else:
        issues["sha256_overlap"] = None

    return issues


def load_combined_metadata() -> pd.DataFrame:
    combined_path = METADATA_ROOT / "combined" / "all_generators_metadata.csv"
    return read_metadata_csv(combined_path)


def load_metadata() -> List[Tuple[str | None, pd.DataFrame, pd.DataFrame]]:
    combined_df = load_combined_metadata()

    if EXPERIMENT_CASE == "combined":
        train_df = combined_df[combined_df["original_split"] == "train"].copy().reset_index(drop=True)
        test_df = combined_df[combined_df["original_split"].isin(["val", "test"])].copy().reset_index(drop=True)
        return [(None, train_df, test_df)]

    if EXPERIMENT_CASE != "cross_generator":
        raise ValueError('EXPERIMENT_CASE phải là "cross_generator" hoặc "combined".')

    generators = sorted(combined_df["generator"].unique().tolist())
    if BASE_GENERATOR is not None:
        query = str(BASE_GENERATOR).lower().strip()
        matched = [g for g in generators if query == g.lower() or query in g.lower()]
        if len(matched) != 1:
            raise ValueError(f"BASE_GENERATOR={BASE_GENERATOR!r} không khớp duy nhất. Generator hợp lệ: {generators}")
        generators = matched

    test_df = combined_df[combined_df["original_split"].isin(["val", "test"])].copy().reset_index(drop=True)
    tasks = []
    for generator in generators:
        train_df = combined_df[
            (combined_df["generator"] == generator) &
            (combined_df["original_split"] == "train")
        ].copy().reset_index(drop=True)
        tasks.append((generator, train_df, test_df))
    return tasks


metadata_tasks = load_metadata()
print("Số task train/evaluate:", len(metadata_tasks))
for base_generator, train_df, test_df in metadata_tasks:
    issues = validate_metadata(train_df, test_df)
    print("base_generator:", base_generator if base_generator else "ALL")
    print("train rows:", len(train_df), "test rows:", len(test_df), "validation:", issues)

Số task train/evaluate: 1
base_generator: imagenet_ai_0419_biggan
train rows: 4000 test rows: 7000 validation: {'train_missing_images': 0, 'train_duplicate_rows': 0, 'test_missing_images': 0, 'test_duplicate_rows': 0, 'image_path_overlap': 0, 'sha256_overlap': 0}


## 6. Dataset và trích xuất feature bằng PyTorch

Phần này là feature engineering chính của Logistic Regression.

Các bước xử lý ảnh:

1. Đọc ảnh bằng PIL và chuyển sang RGB.
2. Resize ảnh về kích thước cố định `IMAGE_SIZE x IMAGE_SIZE` để mọi ảnh có cùng số chiều feature.
3. Dùng `transforms.ToTensor()` để chuyển ảnh từ PIL sang tensor PyTorch và scale pixel từ `[0, 255]` về `[0.0, 1.0]`.
4. Dùng PyTorch để tạo feature:
   - `flat`: vector pixel RGB sau resize, biểu diễn nội dung ảnh ở dạng phẳng.
   - `mean_rgb`: trung bình 3 kênh màu, mô tả xu hướng màu tổng thể.
   - `std_rgb`: độ lệch chuẩn 3 kênh màu, mô tả độ tương phản/phân tán màu.
5. Ghép các feature lại thành một vector duy nhất cho mỗi ảnh.
6. Lưu feature ra `.npz` để lần sau không phải trích xuất lại.

Lưu ý: Đây là baseline đơn giản. Logistic Regression không học trực tiếp từ ảnh dạng tensor 2D mà học từ vector feature đã được flatten và bổ sung thống kê màu.

In [7]:
class ImagePathDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)
        # Resize đưa mọi ảnh về cùng kích thước để số chiều feature cố định.
        # ToTensor chuyển pixel từ [0, 255] về float [0.0, 1.0].
        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
        ])

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        image_tensor = self.transform(image)
        return image_tensor, int(row["label"]), row["image_path"], row["generator"]


def build_dataset(df: pd.DataFrame) -> Dataset:
    return ImagePathDataset(df)


def collate_batch(batch):
    images, labels, paths, generators = zip(*batch)
    return torch.stack(images), np.array(labels), list(paths), list(generators)


def extract_features(df: pd.DataFrame, cache_path: Path) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    if cache_path.exists():
        logger.info("Đọc feature cache: %s", cache_path)
        data = np.load(cache_path, allow_pickle=True)
        features = data["features"]
        labels = data["labels"]
        meta_df = pd.DataFrame(data["meta"].tolist())
        return features, labels, meta_df

    dataset = build_dataset(df)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collate_batch,
    )

    features_list = []
    labels_list = []
    meta_rows = []

    for images, labels, paths, generators in tqdm(loader, desc="Extract features"):
        images = images.to(DEVICE)

        with torch.no_grad():
            # Feature 1: flatten toàn bộ pixel RGB sau resize.
            # Đây là baseline đơn giản, không dùng CNN/deep model để học đặc trưng.
            resized = F.interpolate(images, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
            flat = resized.flatten(start_dim=1)

            # Feature 2: thống kê màu cơ bản theo từng kênh RGB.
            # mean_rgb mô tả màu trung bình, std_rgb mô tả độ phân tán/tương phản màu.
            mean_rgb = images.mean(dim=(2, 3))
            std_rgb = images.std(dim=(2, 3))

            # Ghép pixel feature và color-stat feature thành vector cuối cùng.
            feature = torch.cat([flat, mean_rgb, std_rgb], dim=1)

        features_list.append(feature.cpu().numpy().astype(np.float32))
        labels_list.append(labels.astype(np.int64))

        for path, generator, label in zip(paths, generators, labels):
            meta_rows.append({
                "image_path": path,
                "generator": generator,
                "label": int(label),
            })

    features = np.concatenate(features_list, axis=0)
    labels = np.concatenate(labels_list, axis=0)
    meta_df = pd.DataFrame(meta_rows)

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        cache_path,
        features=features,
        labels=labels,
        meta=meta_df.to_dict(orient="records"),
    )
    logger.info("Đã lưu feature cache: %s", cache_path)
    return features, labels, meta_df

## 7. Train Logistic Regression

Trước khi train Logistic Regression, notebook dùng `StandardScaler`.

Ý nghĩa của `StandardScaler`:

- Với mỗi chiều feature, trừ đi giá trị trung bình của tập train.
- Chia cho độ lệch chuẩn của tập train.
- Giúp các feature có thang đo gần nhau hơn.
- Rất quan trọng với Logistic Regression vì model nhạy với scale của feature.

Quy trình đúng:

1. Fit `StandardScaler` chỉ trên `x_train`.
2. Transform `x_train` bằng scaler đã fit.
3. Train Logistic Regression trên `x_train_scaled`.
4. Khi test, dùng lại đúng scaler đó để transform `x_test`.

Không fit scaler trên test để tránh data leakage.

In [8]:
def train_model(x_train: np.ndarray, y_train: np.ndarray) -> Tuple[StandardScaler, LogisticRegression]:
    # Fit scaler chỉ trên train để tránh leakage từ test.
    # StandardScaler chuẩn hóa từng chiều feature về mean gần 0 và std gần 1.
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)

    # Logistic Regression học boundary tuyến tính trên feature đã chuẩn hóa.
    model = LogisticRegression(
        C=LR_C,
        max_iter=LR_MAX_ITER,
        solver=LR_SOLVER,
        class_weight=LR_CLASS_WEIGHT,
        random_state=LR_RANDOM_STATE,
    )
    model.fit(x_train_scaled, y_train)
    return scaler, model

## 8. Đánh giá model

Model sinh `fake_probability`, tức xác suất ảnh là fake/AI-generated.

Từ xác suất này:

- Nếu `fake_probability >= 0.5`, dự đoán `fake = 1`.
- Nếu `< 0.5`, dự đoán `real = 0`.

Metric được tính:

- Accuracy
- Balanced Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Average Precision
- Confusion Matrix

Ngoài metric tổng thể, notebook còn group prediction theo `generator` để biết model hoạt động tốt/kém trên generator nào.

In [9]:
def evaluate_model(y_true: np.ndarray, y_prob: np.ndarray) -> Dict[str, Any]:
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_prob))
        metrics["average_precision"] = float(average_precision_score(y_true, y_prob))
    else:
        metrics["roc_auc"] = None
        metrics["average_precision"] = None
    return metrics


def evaluate_by_generator(pred_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for generator, part in pred_df.groupby("generator"):
        metrics = evaluate_model(part["label"].to_numpy(), part["fake_probability"].to_numpy())
        metrics["generator"] = generator
        rows.append(metrics)
    return pd.DataFrame(rows)


def plot_confusion_matrix(metrics: Dict[str, Any], path: Path) -> None:
    cm = np.array(metrics["confusion_matrix"])
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def plot_roc(y_true: np.ndarray, y_prob: np.ndarray, path: Path) -> None:
    if len(np.unique(y_true)) < 2:
        return
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr)
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

## 9. Lưu kết quả

Cell này lưu toàn bộ artifact của một lần chạy:

- `scaler.pkl`: scaler đã fit trên train feature.
- `logistic_model.pkl`: Logistic Regression đã train.
- `predictions.csv`: prediction từng ảnh.
- `overall_metrics.json`: metric tổng thể.
- `generator_metrics.csv`: metric theo generator.
- `confusion_matrix.png`: ma trận nhầm lẫn.
- `roc_curve.png`: đường ROC.

Lưu riêng scaler và model để notebook khác có thể tái sử dụng đúng pipeline preprocessing.

In [10]:
def save_results(
    run_dir: Path,
    base_generator: str | None,
    scaler: StandardScaler,
    model: LogisticRegression,
    pred_df: pd.DataFrame,
    overall_metrics: Dict[str, Any],
    generator_metrics_df: pd.DataFrame,
) -> Dict[str, Path]:
    checkpoint_dir = run_dir / "checkpoints"
    prediction_dir = run_dir / "predictions"
    metric_dir = run_dir / "metrics"
    plot_dir = run_dir / "plots"

    scaler_path = checkpoint_dir / "scaler.pkl"
    model_path = checkpoint_dir / "logistic_model.pkl"
    prediction_path = prediction_dir / "predictions.csv"
    overall_metric_path = metric_dir / "overall_metrics.json"
    generator_metric_path = metric_dir / "generator_metrics.csv"
    confusion_matrix_path = plot_dir / "confusion_matrix.png"
    roc_curve_path = plot_dir / "roc_curve.png"

    joblib.dump(scaler, scaler_path)
    joblib.dump(model, model_path)
    pred_df.to_csv(prediction_path, index=False)
    save_json(overall_metrics, overall_metric_path)
    generator_metrics_df.to_csv(generator_metric_path, index=False)
    plot_confusion_matrix(overall_metrics, confusion_matrix_path)
    plot_roc(pred_df["label"].to_numpy(), pred_df["fake_probability"].to_numpy(), roc_curve_path)

    return {
        "scaler": scaler_path,
        "model": model_path,
        "predictions": prediction_path,
        "overall_metrics": overall_metric_path,
        "generator_metrics": generator_metric_path,
        "confusion_matrix": confusion_matrix_path,
        "roc_curve": roc_curve_path,
    }

## 10. Chạy thí nghiệm

Cell này thực hiện toàn bộ pipeline:

1. Load train/test metadata theo `EXPERIMENT_CASE`.
2. Validate metadata và kiểm tra ảnh tồn tại.
3. Trích xuất feature bằng PyTorch.
4. Fit `StandardScaler` trên train feature.
5. Train Logistic Regression.
6. Transform test feature bằng scaler đã fit.
7. Dự đoán xác suất fake.
8. Tính metric tổng thể và metric theo generator.
9. Lưu model, scaler, prediction, metric và biểu đồ.

TH1 `cross_generator`:

- Nếu `BASE_GENERATOR = None`, notebook train riêng từng generator.
- Nếu `BASE_GENERATOR` có tên cụ thể, notebook chỉ train generator đó.

TH2 `combined`:

- Train trên tất cả ảnh có `original_split = train`.
- Test trên `original_split = val` hoặc `test`.

In [11]:
all_results = []

for base_generator, train_df, test_df in metadata_tasks:
    base_name = base_generator if base_generator else "all_generators"
    run_dir = make_output_dir(EXPERIMENT_CASE, base_generator)
    logger.info("Bắt đầu run: experiment=%s, base_generator=%s", EXPERIMENT_CASE, base_name)

    validation = validate_metadata(train_df, test_df)

    train_feature_path = run_dir / "features" / "train_features.npz"
    test_feature_path = run_dir / "features" / "test_features.npz"

    x_train, y_train, _ = extract_features(train_df, train_feature_path)
    x_test, y_test, test_meta_df = extract_features(test_df, test_feature_path)

    scaler, model = train_model(x_train, y_train)
    # Dùng lại scaler đã fit trên train để transform test. Không fit scaler trên test.
    y_prob = model.predict_proba(scaler.transform(x_test))[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    pred_df = test_meta_df.copy()
    pred_df["predicted_label"] = y_pred
    pred_df["fake_probability"] = y_prob
    pred_df["experiment_case"] = EXPERIMENT_CASE
    pred_df["base_generator"] = base_name
    pred_df["model_name"] = "logistic_regression"

    overall_metrics = evaluate_model(y_test, y_prob)
    overall_metrics.update({
        "experiment_case": EXPERIMENT_CASE,
        "base_generator": base_name,
        "feature_extractor": FEATURE_EXTRACTOR,
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "validation": validation,
        "logistic_regression_params": {
            "C": LR_C,
            "max_iter": LR_MAX_ITER,
            "solver": LR_SOLVER,
            "class_weight": LR_CLASS_WEIGHT,
            "random_state": LR_RANDOM_STATE,
        },
    })

    generator_metrics_df = evaluate_by_generator(pred_df)
    output_paths = save_results(run_dir, base_generator, scaler, model, pred_df, overall_metrics, generator_metrics_df)

    result_row = {
        "experiment_case": EXPERIMENT_CASE,
        "base_generator": base_name,
        "run_dir": str(run_dir),
        "accuracy": overall_metrics["accuracy"],
        "balanced_accuracy": overall_metrics["balanced_accuracy"],
        "precision": overall_metrics["precision"],
        "recall": overall_metrics["recall"],
        "f1": overall_metrics["f1"],
        "roc_auc": overall_metrics["roc_auc"],
        "average_precision": overall_metrics["average_precision"],
    }
    all_results.append(result_row)

    print("Run xong:", result_row)
    print("Output paths:")
    for name, path in output_paths.items():
        print(" -", name, path)
    display(generator_metrics_df)

results_df = pd.DataFrame(all_results)
display(results_df)

Extract features:   0%|          | 0/32 [00:00<?, ?it/s]

Extract features:   0%|          | 0/55 [00:00<?, ?it/s]

Run xong: {'experiment_case': 'cross_generator', 'base_generator': 'imagenet_ai_0419_biggan', 'run_dir': '/content/drive/MyDrive/genimage_logistic_regression/outputs/cross_generator/imagenet_ai_0419_biggan/20260720_101941', 'accuracy': 0.536, 'balanced_accuracy': 0.536, 'precision': 0.5489130434782609, 'recall': 0.404, 'f1': 0.46543778801843316, 'roc_auc': 0.5488812244897959, 'average_precision': 0.5679381388701314}
Output paths:
 - scaler /content/drive/MyDrive/genimage_logistic_regression/outputs/cross_generator/imagenet_ai_0419_biggan/20260720_101941/checkpoints/scaler.pkl
 - model /content/drive/MyDrive/genimage_logistic_regression/outputs/cross_generator/imagenet_ai_0419_biggan/20260720_101941/checkpoints/logistic_model.pkl
 - predictions /content/drive/MyDrive/genimage_logistic_regression/outputs/cross_generator/imagenet_ai_0419_biggan/20260720_101941/predictions/predictions.csv
 - overall_metrics /content/drive/MyDrive/genimage_logistic_regression/outputs/cross_generator/imagene

,accuracy,balanced_accuracy,precision,recall,f1,confusion_matrix,roc_auc,average_precision,generator
0,0.741,0.741,0.701843,0.838,0.763902,"[[322, 178], [81, 419]]",0.824892,0.831173,imagenet_ai_0419_biggan
1,0.502,0.502,0.503125,0.322,0.392683,"[[341, 159], [339, 161]]",0.556192,0.507734,imagenet_ai_0419_vqdm
2,0.516,0.516,0.524540,0.342,0.414044,"[[345, 155], [329, 171]]",0.512372,0.535722,imagenet_ai_0424_sdv5
3,0.498,0.498,0.497253,0.362,0.418981,"[[317, 183], [319, 181]]",0.478136,0.518946,imagenet_ai_0424_wukong
4,0.503,0.503,0.504854,0.312,0.385661,"[[347, 153], [344, 156]]",0.502420,0.499675,imagenet_ai_0508_adm
5,0.510,0.510,0.514286,0.360,0.423529,"[[330, 170], [320, 180]]",0.491004,0.490374,imagenet_glide
6,0.482,0.482,0.470968,0.292,0.360494,"[[336, 164], [354, 146]]",0.491080,0.494755,imagenet_midjourney


,experiment_case,base_generator,run_dir,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,cross_generator,imagenet_ai_0419_biggan,/content/drive/MyDrive/genimage_logistic_regre...,0.536,0.536,0.548913,0.404,0.465438,0.548881,0.567938


In [13]:
EXPERIMENT_CASE = "combined"

metadata_tasks = load_metadata()

print("Số task:", len(metadata_tasks))

for base_generator, train_df, test_df in metadata_tasks:
    print("base_generator:", base_generator if base_generator else "all_generators")
    print("train rows:", len(train_df))
    print("test rows:", len(test_df))

Số task: 1
base_generator: all_generators
train rows: 28000
test rows: 7000


In [14]:
all_results = []
if "all_results" not in globals():
    all_results = []

## Chạy TH2

In [15]:
# Chạy riêng TH2 combined, không xóa kết quả TH1 đã có.

EXPERIMENT_CASE = "combined"
metadata_tasks = load_metadata()

if "all_results" not in globals():
    all_results = []

for base_generator, train_df, test_df in metadata_tasks:
    base_name = base_generator if base_generator else "all_generators"
    run_dir = make_output_dir(EXPERIMENT_CASE, base_generator)

    logger.info(
        "Bắt đầu run TH2: experiment=%s, base_generator=%s",
        EXPERIMENT_CASE,
        base_name,
    )

    validation = validate_metadata(train_df, test_df)

    train_feature_path = run_dir / "features" / "train_features.npz"
    test_feature_path = run_dir / "features" / "test_features.npz"

    x_train, y_train, _ = extract_features(train_df, train_feature_path)
    x_test, y_test, test_meta_df = extract_features(test_df, test_feature_path)

    scaler, model = train_model(x_train, y_train)

    # Dùng scaler fit trên train để transform test.
    y_prob = model.predict_proba(scaler.transform(x_test))[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    pred_df = test_meta_df.copy()
    pred_df["predicted_label"] = y_pred
    pred_df["fake_probability"] = y_prob
    pred_df["experiment_case"] = EXPERIMENT_CASE
    pred_df["base_generator"] = base_name
    pred_df["model_name"] = "logistic_regression"

    overall_metrics = evaluate_model(y_test, y_prob)
    overall_metrics.update({
        "experiment_case": EXPERIMENT_CASE,
        "base_generator": base_name,
        "feature_extractor": FEATURE_EXTRACTOR,
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "validation": validation,
        "logistic_regression_params": {
            "C": LR_C,
            "max_iter": LR_MAX_ITER,
            "solver": LR_SOLVER,
            "class_weight": LR_CLASS_WEIGHT,
            "random_state": LR_RANDOM_STATE,
        },
    })

    generator_metrics_df = evaluate_by_generator(pred_df)

    output_paths = save_results(
        run_dir,
        base_generator,
        scaler,
        model,
        pred_df,
        overall_metrics,
        generator_metrics_df,
    )

    result_row = {
        "experiment_case": EXPERIMENT_CASE,
        "base_generator": base_name,
        "run_dir": str(run_dir),
        "accuracy": overall_metrics["accuracy"],
        "balanced_accuracy": overall_metrics["balanced_accuracy"],
        "precision": overall_metrics["precision"],
        "recall": overall_metrics["recall"],
        "f1": overall_metrics["f1"],
        "roc_auc": overall_metrics["roc_auc"],
        "average_precision": overall_metrics["average_precision"],
    }

    all_results.append(result_row)

    print("Run xong TH2:", result_row)
    print("Output paths:")
    for name, path in output_paths.items():
        print(" -", name, path)

    print("Metric TH2 theo từng generator:")
    display(generator_metrics_df)

results_df = pd.DataFrame(all_results)
display(results_df)

Extract features:   0%|          | 0/219 [00:00<?, ?it/s]

Extract features:   0%|          | 0/55 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Run xong TH2: {'experiment_case': 'combined', 'base_generator': 'all_generators', 'run_dir': '/content/drive/MyDrive/genimage_logistic_regression/outputs/combined/all_generators/20260720_102900', 'accuracy': 0.58, 'balanced_accuracy': 0.5800000000000001, 'precision': 0.5803673938002296, 'recall': 0.5777142857142857, 'f1': 0.5790378006872853, 'roc_auc': 0.6084524081632653, 'average_precision': 0.608141377978539}
Output paths:
 - scaler /content/drive/MyDrive/genimage_logistic_regression/outputs/combined/all_generators/20260720_102900/checkpoints/scaler.pkl
 - model /content/drive/MyDrive/genimage_logistic_regression/outputs/combined/all_generators/20260720_102900/checkpoints/logistic_model.pkl
 - predictions /content/drive/MyDrive/genimage_logistic_regression/outputs/combined/all_generators/20260720_102900/predictions/predictions.csv
 - overall_metrics /content/drive/MyDrive/genimage_logistic_regression/outputs/combined/all_generators/20260720_102900/metrics/overall_metrics.json
 - gene

,accuracy,balanced_accuracy,precision,recall,f1,confusion_matrix,roc_auc,average_precision,generator
0,0.701,0.701,0.664484,0.812,0.730873,"[[295, 205], [94, 406]]",0.771792,0.756250,imagenet_ai_0419_biggan
1,0.618,0.618,0.608059,0.664,0.634799,"[[286, 214], [168, 332]]",0.663596,0.625039,imagenet_ai_0419_vqdm
2,0.528,0.528,0.531532,0.472,0.500000,"[[292, 208], [264, 236]]",0.535340,0.548210,imagenet_ai_0424_sdv5
3,0.524,0.524,0.527650,0.458,0.490364,"[[295, 205], [271, 229]]",0.501180,0.561587,imagenet_ai_0424_wukong
4,0.707,0.707,0.667205,0.826,0.738159,"[[294, 206], [87, 413]]",0.760244,0.712529,imagenet_ai_0508_adm
5,0.489,0.489,0.487059,0.414,0.447568,"[[282, 218], [293, 207]]",0.498648,0.486727,imagenet_glide
6,0.493,0.493,0.491358,0.398,0.439779,"[[294, 206], [301, 199]]",0.516768,0.512710,imagenet_midjourney


,experiment_case,base_generator,run_dir,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,combined,all_generators,/content/drive/MyDrive/genimage_logistic_regre...,0.58,0.58,0.580367,0.577714,0.579038,0.608452,0.608141
